In [0]:
%sql

DESCRIBE EXTERNAL LOCATION adb_electrocasa_dev;

name,url,credential_name,owner,created_by,created_at,comment
adb_electrocasa_dev,abfss://unity-catalog-storage@dbstorage2znvi5rhyc2ry.dfs.core.windows.net/7405612877623988,adb_electrocasa_dev,_workspace_admins_adb_electrocasa_dev_7405612877623988,f36e9b6c-238b-4ce4-bb2a-9e11d2b809b2,2026-09-22T04:59:26.308Z,null


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS electrocasa
MANAGED LOCATION 'abfss://unity-catalog-storage@dbstorage2znvi5rhyc2ry.dfs.core.windows.net/7405612877623988/electrocasa'
COMMENT 'Catálogo principal del proyecto ElectroCasa';

CONEXION A TRANCKING ENVIOS


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electrocasa.bronze
COMMENT 'Capa Bronze - datos crudos provenientes de las fuentes';

CREATE SCHEMA IF NOT EXISTS electrocasa.silver
COMMENT 'Capa Silver - datos limpios, estandarizados y deduplicados';

CREATE SCHEMA IF NOT EXISTS electrocasa.gold
COMMENT 'Capa Gold - datos preparados para analítica y negocio';

CREATE VOLUME IF NOT EXISTS electrocasa.bronze.landing
COMMENT 'Landing zone para los archivos fuente del proyecto ElectroCasa';

In [0]:
%sql
SHOW SCHEMAS IN electrocasa;


databaseName
bronze
default
gold
information_schema
silver


In [0]:
%sql
SHOW VOLUMES IN electrocasa.bronze;

database,volume_name
bronze,landing


In [0]:
base_path = "/Volumes/electrocasa/bronze/landing"

carpetas = [
    "ventas",
    "productos",
    "empleados",
    "resenas",
    "devoluciones"
]

for carpeta in carpetas:
    dbutils.fs.mkdirs(f"{base_path}/{carpeta}")

display(dbutils.fs.ls(base_path))

path,name,size,modificationTime
dbfs:/Volumes/electrocasa/bronze/landing/devoluciones/,devoluciones/,0,1790055522000
dbfs:/Volumes/electrocasa/bronze/landing/empleados/,empleados/,0,1790055521000
dbfs:/Volumes/electrocasa/bronze/landing/productos/,productos/,0,1790055521000
dbfs:/Volumes/electrocasa/bronze/landing/resenas/,resenas/,0,1790055522000
dbfs:/Volumes/electrocasa/bronze/landing/ventas/,ventas/,0,1790055521000


In [0]:
base_path = "/Volumes/electrocasa/bronze/landing"

carpetas = [
    "ventas",
    "productos",
    "empleados",
    "resenas",
    "devoluciones"
]

for carpeta in carpetas:
    print(f"\n--- {carpeta.upper()} ---")
    for archivo in dbutils.fs.ls(f"{base_path}/{carpeta}"):
        print(archivo.name)


--- VENTAS ---
ventas_sucursales.csv

--- PRODUCTOS ---
catalogo_productos.json

--- EMPLEADOS ---
empleados_rrhh.csv

--- RESENAS ---
resenas_clientes.json

--- DEVOLUCIONES ---
devoluciones.csv


In [0]:
ruta_ventas = "/Volumes/electrocasa/bronze/landing/ventas/ventas_sucursales.csv"
ruta_productos = "/Volumes/electrocasa/bronze/landing/productos/catalogo_productos.json"
ruta_empleados = "/Volumes/electrocasa/bronze/landing/empleados/empleados_rrhh.csv"
ruta_resenas = "/Volumes/electrocasa/bronze/landing/resenas/resenas_clientes.json"
ruta_devoluciones = "/Volumes/electrocasa/bronze/landing/devoluciones/devoluciones.csv"

print("Rutas configuradas correctamente")

Rutas configuradas correctamente


In [0]:
df_ventas = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ruta_ventas)
)

display(df_ventas.limit(10))

print(f"Total ventas: {df_ventas.count()}")

venta_id,sucursal_id,producto_id,cantidad,monto_total,metodo_pago,fecha_venta,canal
V012083,S08,P00863,4,19852.22,transferencia,2026-06-07,tienda_fisica
V013153,S29,P00861,2,11226.3,Efectivo,2026-04-28,tienda_fisica
V012893,S16,P02271,4,null,TC,2026-05-03,ecommerce
V005386,S17,P01317,4,22300.43,EFECTIVO,2026-07-19,tienda_fisica
V004845,S17,P02591,2,10100.43,Plin,2026-08-23,tienda_fisica
V003122,S03,P02930,1,1778.78,Tarjeta de credito,2026-08-24,tienda_fisica
V002602,S20,P01746,2,2141.28,transferencia,2026-06-23,tienda_fisica
V010330,S24,P01381,1,4857.68,Yape,2026-03-15,tienda_fisica
V002957,S15,P00983,3,9004.36,Transferencia bancaria,2026-05-18,tienda_fisica
V008890,S17,P01812,4,27823.44,tarjeta,2026-04-04,tienda_fisica


Total ventas: 15225


Secret Scope incluso con Lakehouse Federation



In [0]:
%sql

CREATE CONNECTION IF NOT EXISTS electrocasa_sql_tracking
TYPE SQLSERVER
OPTIONS (
    host 'analyticsdmc.database.windows.net',
    port '1433',
    user secret('electrocasa-sql', 'sql-user'),
    password secret('electrocasa-sql', 'sql-password')
)
COMMENT 'Conexión federada a Azure SQL para TrackingEnvios del proyecto ElectroCasa';



In [0]:
%sql

SHOW CONNECTIONS;

name,type,comment,owner,read_only,options,properties,created_by,created_at,updated_by,updated_at,url
electrocasa_sql_tracking,SQLSERVER,Conexión federada a Azure SQL para TrackingEnvios del proyecto ElectroCasa,loayzaindira@gmail.com,true,"host -> analyticsdmc.database.windows.net, port -> 1433",,loayzaindira@gmail.com,2026-09-22 AD at 06:39:09 UTC,loayzaindira@gmail.com,2026-09-22 AD at 06:39:09 UTC,jdbc://analyticsdmc.database.windows.net:1433/


In [0]:
%sql

DESCRIBE CONNECTION electrocasa_sql_tracking;

info_name,info_value
Connection Name,electrocasa_sql_tracking
Type,SQLSERVER
Comment,Conexión federada a Azure SQL para TrackingEnvios del proyecto ElectroCasa
Owner,loayzaindira@gmail.com
Read-only,true
Options,"host -> analyticsdmc.database.windows.net, port -> 1433"


ahora falta vincular esa conexión con la base de datos Azure SQL que les proporcionaron

In [0]:
%sql

CREATE FOREIGN CATALOG IF NOT EXISTS electrocasa_tracking
USING CONNECTION electrocasa_sql_tracking
OPTIONS (
    database 'electrocasadb'
)


In [0]:
%sql

SHOW CATALOGS;

catalog
adb_electrocasa_dev
electrocasa
electrocasa_tracking
samples
system


In [0]:
import socket

host = "analyticsdmc.database.windows.net"
port = 1433

try:
    with socket.create_connection((host, port), timeout=10):
        print("OK - Databricks puede alcanzar Azure SQL por el puerto 1433")
except Exception as e:
    print("ERROR DE CONECTIVIDAD:")
    print(e)

OK - Databricks puede alcanzar Azure SQL por el puerto 1433


In [0]:
%sql
SHOW TABLES IN electrocasa_tracking.dbo;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7844940214414222>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SHOW TABLES IN electrocasa_tracking.dbo;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 except BaseException as e:
    208     self.driver_activity_logg

In [0]:
%sql

SHOW TABLES IN electrocasa_tracking.dbo;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7844940214414221>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', '\nSHOW TABLES IN electrocasa_tracking.dbo;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 except BaseException as e:
    208     self.driver_activity_lo

In [0]:
%sql

SHOW CONNECTIONS;

name,type,comment,owner,read_only,options,properties,created_by,created_at,updated_by,updated_at,url
electrocasa_sql_tracking,SQLSERVER,Conexión federada a Azure SQL para TrackingEnvios del proyecto ElectroCasa,loayzaindira@gmail.com,true,"host -> analyticsdmc.database.windows.net, port -> 1433",,loayzaindira@gmail.com,2026-09-22 AD at 06:39:09 UTC,loayzaindira@gmail.com,2026-09-22 AD at 06:39:09 UTC,jdbc://analyticsdmc.database.windows.net:1433/


In [0]:
import socket

host = "analyticsdmc.database.windows.net"
port = 1433

try:
    with socket.create_connection((host, port), timeout=10):
        print("OK - conexión TCP al servidor Azure SQL por puerto 1433")
except Exception as e:
    print("ERROR - no se pudo conectar:")
    print(type(e).__name__, str(e))

OK - conexión TCP al servidor Azure SQL por puerto 1433


In [0]:
%sql

DESCRIBE CATALOG EXTENDED electrocasa_tracking;

info_name,info_value
Catalog Name,electrocasa_tracking
Comment,
Owner,loayzaindira@gmail.com
Catalog Type,Foreign
Connection Name,electrocasa_sql_tracking
Options,database = electrocasadb
Created By,loayzaindira@gmail.com
Created At,2026-09-22 AD at 06:45:36 UTC
Updated By,loayzaindira@gmail.com
Updated At,2026-09-22 AD at 06:45:36 UTC


In [0]:
%sql
DESCRIBE CATALOG EXTENDED electrocasa_tracking;

info_name,info_value
Catalog Name,electrocasa_tracking
Comment,
Owner,loayzaindira@gmail.com
Catalog Type,Foreign
Connection Name,electrocasa_sql_tracking
Options,database = electrocasadb
Created By,loayzaindira@gmail.com
Created At,2026-09-22 AD at 06:45:36 UTC
Updated By,loayzaindira@gmail.com
Updated At,2026-09-22 AD at 06:45:36 UTC


In [0]:
user = dbutils.secrets.get(
    scope="electrocasa-sql",
    key="sql-user"
)

password = dbutils.secrets.get(
    scope="electrocasa-sql",
    key="sql-password"
)

jdbc_url = (
    "jdbc:sqlserver://analyticsdmc.database.windows.net:1433;"
    "database=electrocasadb;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "loginTimeout=30;"
)

try:
    df_test = (
        spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("query", "SELECT 1 AS prueba")
        .option("user", user)
        .option("password", password)
        .load()
    )

    display(df_test)

except Exception as e:
    print("ERROR JDBC:")
    print(str(e)[:1500])

ERROR JDBC:
[FAILED_JDBC.CONNECTION] Failed JDBC jdbc:sqlserver:*********(redacted) on the operation: Failed to connect to the database. SQLSTATE: HV000

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.jdbc.NoLegacyJDBCError.classifyException(JdbcDialects.scala:1100)
	at org.apache.spark.sql.jdbc.NoLegacyJDBCError.classifyException$(JdbcDialects.scala:1085)
	at org.apache.spark.sql.jdbc.MsSqlServerDialect.classifyException(MsSqlServerDialect.scala:392)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.classifyException(JdbcUtils.scala:1316)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:58)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:107)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createConnectionFactory$2(JdbcDialects.scala:302)
	at org.apache.spark.sql.catalyst.MetricKeyUtils$

In [0]:
%sql

SELECT *
FROM remote_query(
  'electrocasa_sql_tracking',
  database => 'electrocasadb',
  query => 'SELECT 1 AS prueba'
);

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8945399402119590>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "\nSELECT *\nFROM remote_query(\n  'electrocasa_sql_tracking',\n  database => 'electrocasadb',\n  query => 'SELECT 1 AS prueba'\n);\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(

In [0]:
username = dbutils.secrets.get(
    scope="electrocasa-sql",
    key="sql-user"
)

password = dbutils.secrets.get(
    scope="electrocasa-sql",
    key="sql-password"
)

jdbc_url = (
    "jdbc:sqlserver://analyticsdmc.database.windows.net:1433;"
    "database=electrocasadb;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "loginTimeout=30;"
)

In [0]:
columnas_tracking = """
tracking_id,
pedido_id,
courier,
estado_entrega,
sucursal_origen,
fecha_actualizacion
"""

df_tracking = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option(
        "dbtable",
        f"""
        (
            SELECT
                {columnas_tracking}
            FROM dbo.TrackingEnvios
        ) AS tracking
        """
    )
    .option("user", username)
    .option("password", password)
    .option("fetchsize", "1000")
    .load()
)

display(df_tracking.limit(10))

tracking_id,pedido_id,courier,estado_entrega,sucursal_origen,fecha_actualizacion
TRK000572,PED004794,servientrega,Pendiente,S34,2026-01-18
TRK000628,PED009832,OLVA COURIER,en_transito,S19,2026-07-26
TRK002601,PED002771,Olva Courier,DEVUELTO,S36,2026-06-04
TRK004555,PED009300,servientrega,Pendiente,S28,2026-07-16
TRK003609,PED016634,Shalom,entregado,S03,2026-04-25
TRK001925,PED005290,Servientrega,DEVUELTO,S13,2026-08-16
TRK002784,PED018335,Olva Courier,EN_CAMINO,S05,2026-05-01
TRK002817,PED008324,Servientrega,entregado,S15,2026-06-20
TRK001671,PED005251,servientrega,En camino,S10,2026-08-26
TRK003389,PED010302,servientrega,En camino,S33,2026-04-03


In [0]:
%sql

DROP CONNECTION IF EXISTS prueba_tracking;

In [0]:
%sql

SHOW CONNECTIONS;

name,type,comment,owner,read_only,options,properties,created_by,created_at,updated_by,updated_at,url
electrocasa_sql_tracking,SQLSERVER,Conexión federada a Azure SQL para TrackingEnvios del proyecto ElectroCasa,loayzaindira@gmail.com,true,"host -> analyticsdmc.database.windows.net, port -> 1433",,loayzaindira@gmail.com,2026-09-22 AD at 06:39:09 UTC,loayzaindira@gmail.com,2026-09-22 AD at 06:39:09 UTC,jdbc://analyticsdmc.database.windows.net:1433/


In [0]:
%sql

SHOW CATALOGS;

catalog
adb_electrocasa_dev
electrocasa
electrocasa_tracking
samples
system


In [0]:
%sql

SHOW SCHEMAS IN electrocasa_tracking;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8945399402119600>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', '\nSHOW SCHEMAS IN electrocasa_tracking;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 except BaseException as e:
    208     self.driver_activity_logge

In [0]:
%sql

SHOW CATALOGS;

catalog
adb_electrocasa_dev
electrocasa
samples
system


In [0]:
%sql

SHOW CONNECTIONS;

name,type,comment,owner,read_only,options,properties,created_by,created_at,updated_by,updated_at,url


*******CONEXION TRACKING ENVIOS CON SECRET SCOPE

In [0]:
username = dbutils.secrets.get(
    scope="electrocasa-sql",
    key="sql-user"
)

password = dbutils.secrets.get(
    scope="electrocasa-sql",
    key="sql-password"
)

jdbc_url = (
    "jdbc:sqlserver://analyticsdmc.database.windows.net:1433;"
    "database=electrocasadb;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "loginTimeout=30;"
)

In [0]:
df_tracking = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option(
        "dbtable",
        """
        (
            SELECT
                tracking_id,
                pedido_id,
                courier,
                estado_entrega,
                sucursal_origen,
                fecha_actualizacion
            FROM dbo.TrackingEnvios
        ) AS tracking
        """
    )
    .option("user", username)
    .option("password", password)
    .option("fetchsize", "1000")
    .load()
)

In [0]:
display(df_tracking.limit(10))

tracking_id,pedido_id,courier,estado_entrega,sucursal_origen,fecha_actualizacion
TRK000572,PED004794,servientrega,Pendiente,S34,2026-01-18
TRK000628,PED009832,OLVA COURIER,en_transito,S19,2026-07-26
TRK002601,PED002771,Olva Courier,DEVUELTO,S36,2026-06-04
TRK004555,PED009300,servientrega,Pendiente,S28,2026-07-16
TRK003609,PED016634,Shalom,entregado,S03,2026-04-25
TRK001925,PED005290,Servientrega,DEVUELTO,S13,2026-08-16
TRK002784,PED018335,Olva Courier,EN_CAMINO,S05,2026-05-01
TRK002817,PED008324,Servientrega,entregado,S15,2026-06-20
TRK001671,PED005251,servientrega,En camino,S10,2026-08-26
TRK003389,PED010302,servientrega,En camino,S33,2026-04-03


In [0]:
%sql

SHOW CONNECTIONS;


name,type,comment,owner,read_only,options,properties,created_by,created_at,updated_by,updated_at,url
electrocasa_sql_tracking,SQLSERVER,Conexión federada a Azure SQL para TrackingEnvios,loayzaindira@gmail.com,true,"host -> analyticsdmc.database.windows.net, port -> 1433",,loayzaindira@gmail.com,2026-09-23 AD at 03:40:30 UTC,loayzaindira@gmail.com,2026-09-23 AD at 03:40:30 UTC,jdbc://analyticsdmc.database.windows.net:1433/


In [0]:
%sql

DESCRIBE CONNECTION electrocasa_sql_tracking;

info_name,info_value
Connection Name,electrocasa_sql_tracking
Type,SQLSERVER
Comment,Conexión federada a Azure SQL para TrackingEnvios
Owner,loayzaindira@gmail.com
Read-only,true
Options,"host -> analyticsdmc.database.windows.net, port -> 1433"


In [0]:
%sql

SHOW SCHEMAS IN electrocasa_tracking;

databaseName
db_accessadmin
db_backupoperator
db_datareader
db_datawriter
db_ddladmin
db_denydatareader
db_denydatawriter
db_owner
db_securityadmin
dbo


In [0]:
%sql

SHOW TABLES IN electrocasa_tracking.dbo;

database,tableName,isTemporary
dbo,trackingenvios,false


In [0]:
%sql

SELECT *
FROM electrocasa_tracking.dbo.trackingenvios
LIMIT 10;

tracking_id,pedido_id,courier,estado_entrega,sucursal_origen,fecha_actualizacion
TRK000572,PED004794,servientrega,Pendiente,S34,2026-01-18
TRK000628,PED009832,OLVA COURIER,en_transito,S19,2026-07-26
TRK002601,PED002771,Olva Courier,DEVUELTO,S36,2026-06-04
TRK004555,PED009300,servientrega,Pendiente,S28,2026-07-16
TRK003609,PED016634,Shalom,entregado,S03,2026-04-25
TRK001925,PED005290,Servientrega,DEVUELTO,S13,2026-08-16
TRK002784,PED018335,Olva Courier,EN_CAMINO,S05,2026-05-01
TRK002817,PED008324,Servientrega,entregado,S15,2026-06-20
TRK001671,PED005251,servientrega,En camino,S10,2026-08-26
TRK003389,PED010302,servientrega,En camino,S33,2026-04-03


ojooo

In [0]:
from importlib.metadata import version
from databricks.sdk import WorkspaceClient

print("Versión SDK:", version("databricks-sdk"))

w = WorkspaceClient()

print("\nMétodos disponibles en workspace_iam_v2:")
for metodo in dir(w.workspace_iam_v2):
    if "group" in metodo.lower():
        print(metodo)

Versión SDK: 0.67.0

Métodos disponibles en workspace_iam_v2:
resolve_group_proxy


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

print("APIs relacionadas con grupos disponibles en WorkspaceClient:\n")

for atributo in dir(w):
    if "group" in atributo.lower():
        print(atributo)

APIs relacionadas con grupos disponibles en WorkspaceClient:

_groups
_groups_v2
groups
groups_v2


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

print("Métodos disponibles en groups_v2:\n")

for metodo in dir(w.groups_v2):
    if not metodo.startswith("_"):
        print(metodo)

Métodos disponibles en groups_v2:

create
delete
get
list
patch
update


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

grupos = list(w.groups_v2.list())

for grupo in grupos:
    print(
        "Nombre:", grupo.display_name,
        "| ID:", grupo.id,
        "| Meta:", grupo.meta
    )

Nombre: admins | ID: 150776615900729 | Meta: ResourceMeta(resource_type='WorkspaceGroup')
Nombre: users | ID: 156074792869255 | Meta: ResourceMeta(resource_type='WorkspaceGroup')


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

resultado = w.api_client.do(
    "GET",
    "/api/2.0/identity/groups"
)

print(resultado)

{'groups': [{'account_id': 'e57faddf-4f7d-4921-a925-3a709aae8349', 'group_id': '151821082871387', 'group_name': 'account users'}]}


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

grupo = w.api_client.do(
    "POST",
    "/api/2.0/identity/groups",
    body={
        "group_name": "electrocasa_ingenieria"
    }
)

print(grupo)

{'account_id': 'e57faddf-4f7d-4921-a925-3a709aae8349', 'group_id': '149799763489333', 'group_name': 'electrocasa_ingenieria'}


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

grupos_requeridos = [
    "electrocasa_ingenieria",
    "electrocasa_analistas",
    "electrocasa_auditoria"
]

respuesta = w.api_client.do(
    "GET",
    "/api/2.0/identity/groups"
)

grupos_existentes = {
    grupo["group_name"]
    for grupo in respuesta.get("groups", [])
}

for nombre_grupo in grupos_requeridos:
    if nombre_grupo in grupos_existentes:
        print(f"YA EXISTE: {nombre_grupo}")
    else:
        creado = w.api_client.do(
            "POST",
            "/api/2.0/identity/groups",
            body={
                "group_name": nombre_grupo
            }
        )

        print(
            f"CREADO: {creado['group_name']} "
            f"| ID: {creado['group_id']}"
        )

YA EXISTE: electrocasa_ingenieria
CREADO: electrocasa_analistas | ID: 152154022255228
CREADO: electrocasa_auditoria | ID: 154964525503034


3 grupos 


In [0]:
respuesta = w.api_client.do(
    "GET",
    "/api/2.0/identity/groups"
)

for grupo in respuesta.get("groups", []):
    if grupo["group_name"].startswith("electrocasa_"):
        print(
            grupo["group_name"],
            "| ID:",
            grupo["group_id"]
        )

electrocasa_analistas | ID: 152154022255228
electrocasa_auditoria | ID: 154964525503034
electrocasa_ingenieria | ID: 149799763489333


ver que existe GRANT/REVOKE


In [0]:
%sql

SHOW GRANTS ON CATALOG electrocasa;

Principal,ActionType,ObjectType,ObjectKey


In [0]:
%sql

GRANT USE CATALOG
ON CATALOG electrocasa
TO `electrocasa_ingenieria`;

GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE
ON SCHEMA electrocasa.bronze
TO `electrocasa_ingenieria`;

GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE
ON SCHEMA electrocasa.silver
TO `electrocasa_ingenieria`;

GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE
ON SCHEMA electrocasa.gold
TO `electrocasa_ingenieria`;

In [0]:
%sql

GRANT USE CATALOG
ON CATALOG electrocasa
TO electrocasa_ingenieria;

In [0]:
%sql

SHOW GRANTS electrocasa_ingenieria
ON CATALOG electrocasa;

Principal,ActionType,ObjectType,ObjectKey
electrocasa_ingenieria,USE CATALOG,CATALOG,electrocasa


In [0]:
%sql

GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE
ON SCHEMA electrocasa.bronze
TO electrocasa_ingenieria;

GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE
ON SCHEMA electrocasa.silver
TO electrocasa_ingenieria;

GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE
ON SCHEMA electrocasa.gold
TO electrocasa_ingenieria;

In [0]:
%sql

GRANT READ VOLUME, WRITE VOLUME
ON SCHEMA electrocasa.bronze
TO electrocasa_ingenieria;

In [0]:
%sql

SHOW GRANTS electrocasa_ingenieria
ON SCHEMA electrocasa.bronze;

Principal,ActionType,ObjectType,ObjectKey
electrocasa_ingenieria,READ VOLUME,SCHEMA,electrocasa.bronze
electrocasa_ingenieria,CREATE TABLE,SCHEMA,electrocasa.bronze
electrocasa_ingenieria,SELECT,SCHEMA,electrocasa.bronze
electrocasa_ingenieria,MODIFY,SCHEMA,electrocasa.bronze
electrocasa_ingenieria,WRITE VOLUME,SCHEMA,electrocasa.bronze
electrocasa_ingenieria,USE SCHEMA,SCHEMA,electrocasa.bronze


In [0]:
%sql

SHOW GRANTS electrocasa_ingenieria
ON SCHEMA electrocasa.silver;

Principal,ActionType,ObjectType,ObjectKey
electrocasa_ingenieria,CREATE TABLE,SCHEMA,electrocasa.silver
electrocasa_ingenieria,MODIFY,SCHEMA,electrocasa.silver
electrocasa_ingenieria,SELECT,SCHEMA,electrocasa.silver
electrocasa_ingenieria,USE SCHEMA,SCHEMA,electrocasa.silver


In [0]:
%sql

SHOW GRANTS electrocasa_ingenieria
ON SCHEMA electrocasa.gold;

Principal,ActionType,ObjectType,ObjectKey
electrocasa_ingenieria,CREATE TABLE,SCHEMA,electrocasa.gold
electrocasa_ingenieria,MODIFY,SCHEMA,electrocasa.gold
electrocasa_ingenieria,SELECT,SCHEMA,electrocasa.gold
electrocasa_ingenieria,USE SCHEMA,SCHEMA,electrocasa.gold


Analistas -- solo lectura sobre Gold

In [0]:
%sql

GRANT USE CATALOG
ON CATALOG electrocasa
TO electrocasa_analistas;

GRANT USE SCHEMA, SELECT
ON SCHEMA electrocasa.gold
TO electrocasa_analistas;

Añadir REVOKE

In [0]:
%sql

REVOKE ALL PRIVILEGES
ON SCHEMA electrocasa.bronze
FROM electrocasa_analistas;

REVOKE ALL PRIVILEGES
ON SCHEMA electrocasa.silver
FROM electrocasa_analistas;

validar


In [0]:
%sql

SHOW GRANTS electrocasa_analistas
ON CATALOG electrocasa;

Principal,ActionType,ObjectType,ObjectKey
electrocasa_analistas,USE CATALOG,CATALOG,electrocasa


In [0]:
%sql

SHOW GRANTS electrocasa_analistas
ON SCHEMA electrocasa.gold;

Principal,ActionType,ObjectType,ObjectKey
electrocasa_analistas,SELECT,SCHEMA,electrocasa.gold
electrocasa_analistas,USE SCHEMA,SCHEMA,electrocasa.gold


Auditoría. solo lectura sobre Gold y además acceso a evidencia de auditoría/lineage.

In [0]:
%sql

GRANT USE CATALOG
ON CATALOG electrocasa
TO electrocasa_auditoria;

GRANT BROWSE
ON CATALOG electrocasa
TO electrocasa_auditoria;

GRANT USE SCHEMA, SELECT
ON SCHEMA electrocasa.gold
TO electrocasa_auditoria;

In [0]:
%sql

REVOKE ALL PRIVILEGES
ON SCHEMA electrocasa.bronze
FROM electrocasa_auditoria;

REVOKE ALL PRIVILEGES
ON SCHEMA electrocasa.silver
FROM electrocasa_auditoria;

validando


In [0]:
%sql

SHOW GRANTS electrocasa_auditoria
ON CATALOG electrocasa;

Principal,ActionType,ObjectType,ObjectKey
electrocasa_auditoria,BROWSE,CATALOG,electrocasa
electrocasa_auditoria,USE CATALOG,CATALOG,electrocasa


In [0]:
%sql

SHOW GRANTS electrocasa_auditoria
ON SCHEMA electrocasa.gold;

Principal,ActionType,ObjectType,ObjectKey
electrocasa_auditoria,SELECT,SCHEMA,electrocasa.gold
electrocasa_auditoria,USE SCHEMA,SCHEMA,electrocasa.gold


proteger dni y salario.

In [0]:
%sql

DESCRIBE TABLE electrocasa.silver.empleados;

col_name,data_type,comment
id_empleado,string,null
nombre,string,null
dni,string,null
email,string,null
salario,double,null
sucursal_id,string,null
cargo,string,null
tipo_evento,string,null
fecha_evento,date,null
_rescued_data,string,null


In [0]:
%sql

SELECT
  current_user() AS usuario_actual,
  is_account_group_member('electrocasa_ingenieria') AS pertenece_ingenieria;

usuario_actual,pertenece_ingenieria
loayzaindira@gmail.com,false


masking


In [0]:
%sql

CREATE OR REPLACE FUNCTION electrocasa.silver.mask_dni(valor STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_account_group_member('electrocasa_ingenieria')
      THEN valor
    WHEN valor IS NULL
      THEN NULL
    ELSE '********'
  END;

CREATE OR REPLACE FUNCTION electrocasa.silver.mask_salario(valor DOUBLE)
RETURNS DOUBLE
RETURN
  CASE
    WHEN is_account_group_member('electrocasa_ingenieria')
      THEN valor
    ELSE CAST(NULL AS DOUBLE)
  END;

In [0]:
%sql

ALTER MATERIALIZED VIEW electrocasa.silver.empleados
ALTER COLUMN dni
SET MASK electrocasa.silver.mask_dni;

In [0]:
%sql

SELECT
  id_empleado,
  dni
FROM electrocasa.silver.empleados
LIMIT 10;

id_empleado,dni
E00760,********
E01154,********
E00335,********
E01290,********
E01495,********
E01087,********
E01543,********
E00979,********
E00981,********
E00646,********


In [ ]:
%sql

CREATE CONNECTION IF NOT EXISTS electrocasa_sql_tracking
TYPE SQLSERVER
OPTIONS (
    host 'analyticsdmc.database.windows.net',
    port '1433',
    user secret('electrocasa-sql', 'sql-user'),
    password secret('electrocasa-sql', 'sql-password')
)
COMMENT 'Conexión federada a Azure SQL para TrackingEnvios del proyecto ElectroCasa';

In [ ]:
%sql

CREATE FOREIGN CATALOG IF NOT EXISTS electrocasa_tracking
USING CONNECTION electrocasa_sql_tracking
OPTIONS (
    database 'electrocasadb'
);